In [39]:
import os
import sys
import time
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# SHARED CONFIGURATION PARAMETERS
# =============================================================================

# -- File & Path Configuration --
HDF_FILE_PATH = '../../data/raw/all_hourly_data.h5'
FEATURE_CLASSIFICATION_PATH = '../../data/processed/eda_results_corrected/feature_classification.csv'
OUTPUT_DIR = 'phase_1_outputs'

# -- Experiment Configuration --
DRY_RUN = False  # Set to False for full dataset
DRY_RUN_PATIENTS = 2000  # Number of patients for dry run
USE_CACHED_PREPROCESSING = True  # Use cache if available

# -- Feature Engineering Options --
CALCULATE_TRENDS = True  # Include trend calculations
WINDOW_SIZE = 24  # Hours of data for feature extraction
GAP_TIME = 6      # Hours of gap before prediction

# -- Target and Reproducibility --
TARGET_VARIABLE = 'mort_hosp'
SEED = 42

# -- Hyperparameter Tuning Configuration --
N_OPTUNA_TRIALS = 15  # Number of trials for hyperparameter search
OPTUNA_TIMEOUT = 1800 # Timeout in seconds

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [38]:
print("=" * 70)
print("PHASE 1: CLEAN DATA PREPROCESSING")
print("=" * 70)

preprocessing_start = time.time()

# Import and run preprocessing
try:
    # Import the clean preprocessing script as a module
    import importlib.util
    spec = importlib.util.spec_from_file_location("data_preprocessing_clean", "data_preprocessing_clean.py")
    if spec is None:
        raise ImportError("Could not load data_preprocessing_clean.py")
    
    preprocessing_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for data_preprocessing_clean.py")
    
    # Execute the preprocessing script
    print(f"Starting clean data preprocessing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    spec.loader.exec_module(preprocessing_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'HDF_FILE_PATH': HDF_FILE_PATH,
        'FEATURE_CLASSIFICATION_PATH': FEATURE_CLASSIFICATION_PATH,
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'USE_CACHED_PREPROCESSING': USE_CACHED_PREPROCESSING,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED
    }
    
    # Call the main function explicitly with configuration
    preprocessing_module.main(config)
    
    preprocessing_time = time.time() - preprocessing_start
    print(f"\n✓ Clean data preprocessing completed in {preprocessing_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during preprocessing: {e}")
    raise


2025-07-01 21:11:06,462 [INFO] - Logging initialized. Output directory: phase_1_outputs
2025-07-01 21:11:06,463 [INFO] - Starting data loading from: ../../data/raw/all_hourly_data.h5


PHASE 1: CLEAN DATA PREPROCESSING
Starting clean data preprocessing at 2025-07-01 21:11:06


2025-07-01 21:11:09,585 [INFO] - ✓ Data loaded: patients=(34472, 28), time-series=(2200954, 104)
2025-07-01 21:11:09,591 [INFO] - ✓ Patients DataFrame structure:
2025-07-01 21:11:09,592 [INFO] -   - Index levels: ['subject_id', 'hadm_id', 'icustay_id']
2025-07-01 21:11:09,593 [INFO] -   - Columns: ['gender', 'ethnicity', 'age', 'insurance', 'admittime', 'diagnosis_at_admission', 'dischtime', 'discharge_location', 'fullcode_first', 'dnr_first', 'fullcode', 'dnr', 'dnr_first_charttime', 'cmo_first', 'cmo_last', 'cmo', 'deathtime', 'intime', 'outtime', 'los_icu', 'admission_type', 'first_careunit', 'mort_icu', 'mort_hosp', 'hospital_expire_flag', 'hospstay_seq', 'readmission_30', 'max_hours']
2025-07-01 21:11:09,593 [INFO] -   - Shape: (34472, 28)
2025-07-01 21:11:09,594 [INFO] - Extracting demographic features...
2025-07-01 21:12:17,431 [INFO] - ✓ Extracted demographic features: (34472, 7)
2025-07-01 21:12:17,433 [INFO] -   - Age range: 15.1 - 310.3
2025-07-01 21:12:17,435 [INFO] -   - G

In [31]:
print("=" * 70)
print("PHASE 2: XGBOOST MODEL TRAINING & EVALUATION")
print("=" * 70)

xgboost_start = time.time()

# Initialize variables for results capture
auroc_results = None
auprc_results = None
final_model = None
xgboost_results = None

try:
    # Import and run XGBoost analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("xgboost_analysis", "xgboost_analysis.py")
    if spec is None:
        raise ImportError("Could not load xgboost_analysis.py")
    
    xgboost_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for xgboost_analysis.py")
    
    print(f"Starting XGBoost analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the XGBoost analysis module and call main function
    spec.loader.exec_module(xgboost_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function explicitly with configuration
    xgboost_module.main(config)
    
    # Load the results that were saved by the XGBoost script
    results_path = os.path.join(OUTPUT_DIR, 'results_xgboost_baseline.pkl')
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            xgboost_results = pickle.load(f)
        
        # Extract key metrics for display
        auroc_results = {
            'point_estimate': xgboost_results['test_auroc'],
            'ci_lower': xgboost_results['test_auroc_ci_lower'],
            'ci_upper': xgboost_results['test_auroc_ci_upper']
        }
        
        auprc_results = {
            'point_estimate': xgboost_results['test_auprc'], 
            'ci_lower': xgboost_results['test_auprc_ci_lower'],
            'ci_upper': xgboost_results['test_auprc_ci_upper']
        }
        
        print(f"\n✅ XGBoost Results Successfully Captured:")
        print(f"   AUROC: {auroc_results['point_estimate']:.4f} "
              f"(95% CI: {auroc_results['ci_lower']:.4f}-{auroc_results['ci_upper']:.4f})")
        print(f"   AUPRC: {auprc_results['point_estimate']:.4f} "
              f"(95% CI: {auprc_results['ci_lower']:.4f}-{auprc_results['ci_upper']:.4f})")
    else:
        print("⚠️  Results file not found - metrics may not be available in summary")
    
    xgboost_time = time.time() - xgboost_start
    print(f"\n✓ XGBoost analysis completed in {xgboost_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during XGBoost analysis: {e}")
    print(f"   This may be due to XGBoost library installation issues")
    print(f"   Preprocessing completed successfully - data is ready for manual analysis")
    xgboost_time = time.time() - xgboost_start
    print(f"   Attempted runtime: {xgboost_time/60:.2f} minutes")
    # Don't raise - allow pipeline to continue and show preprocessing results


2025-07-01 20:59:30,852 [INFO] - Loading preprocessed data from cache...
2025-07-01 20:59:30,861 [INFO] - ✓ Successfully loaded preprocessed data with prefix: preprocessed_mort_hosp_dryrun_1000_trends_True_window_24_gap_6_seed_42
2025-07-01 20:59:30,862 [INFO] - Data shapes: X_train=(452, 452), X_val=(65, 452), X_test=(173, 452)
2025-07-01 20:59:30,863 [INFO] - Performing additional data cleaning for XGBoost...
2025-07-01 20:59:30,863 [INFO] - Before cleaning - NaN counts:
2025-07-01 20:59:30,866 [INFO] -   X_train: 60541
2025-07-01 20:59:30,867 [INFO] -   X_val: 8542
2025-07-01 20:59:30,869 [INFO] -   X_test: 23458
2025-07-01 20:59:30,869 [INFO] - Before cleaning - Inf counts:
2025-07-01 20:59:30,871 [INFO] -   X_train: 0
2025-07-01 20:59:30,872 [INFO] -   X_val: 0
2025-07-01 20:59:30,874 [INFO] -   X_test: 0
2025-07-01 20:59:30,911 [INFO] - Imputed extremely sparse feature: albumin pleural_mean_last (>95% missing)
2025-07-01 20:59:30,913 [INFO] - Imputed extremely sparse feature: cre

PHASE 2: XGBOOST MODEL TRAINING & EVALUATION
Starting XGBoost analysis at 2025-07-01 20:59:30


/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  UserWarning,
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  UserWarning,
[I 2025-07-01 20:59:31,555] Trial 0 finished with value: 0.8559322033898306 and parameters: {'n_estimators': 750, 'learning_rate': 0.002280009741912104, 'max_depth': 10, 'subsample': 0.689562196548416, 'colsample_bytree': 0.9368479354992695, 'gamma': 9.62278253667906e-07, 'min_child_weight': 8}. Best is trial 0 with value: 0.8559322033898306.
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797

              precision    recall  f1-score   support

           0       0.91      0.97      0.94       156
           1       0.29      0.12      0.17        17

    accuracy                           0.88       173
   macro avg       0.60      0.54      0.55       173
weighted avg       0.85      0.88      0.86       173

[[151   5]
 [ 15   2]]

✅ XGBoost Results Successfully Captured:
   AUROC: 0.7017 (95% CI: 0.5512-0.8225)
   AUPRC: 0.2920 (95% CI: 0.1550-0.4729)

✓ XGBoost analysis completed in 0.11 minutes
